# 01 - Exploratory Data Analysis (EDA)
### JoSAA Counselling Cutoffs (2021 - 2025)
**Platform:** Vikalp Admission-Probability Engine
**Objective:** Characterize cutoff distributions, missingness patterns, cutoff drift dynamics, and category quota scaling.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt

# Ensure root is in path
root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

DATA_PATH = root_dir / "data" / "processed" / "cutoffs.parquet"
df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df):,} total cutoff records.")


## 1. Dataset Overview & Annual Offerings
Examine the record count by year and verify that JoSAA cycles 2021 through 2025 are present.


In [ ]:
yearly_counts = df["year"].value_counts().sort_index()
print("Record count by JoSAA cycle:")
print(yearly_counts)

institutes_per_year = df.groupby("year")["institute"].nunique()
programs_per_year = df.groupby("year")["program"].nunique()

summary_by_year = pd.DataFrame({
    "Records": yearly_counts,
    "Institutes": institutes_per_year,
    "Programs": programs_per_year,
})
summary_by_year


## 2. Missingness Analysis & Sparse History
Quantify how many offerings have full 3-year history vs. sparse history (< 3 years of lags).


In [ ]:
df_2025 = df[df["year"] == 2025].copy()
total_2025 = len(df_2025)

has_lag1 = df_2025["prev_closing_1"].notna().sum()
has_lag2 = df_2025["prev_closing_2"].notna().sum()
has_lag3 = df_2025["prev_closing_3"].notna().sum()

print(f"2025 Total Offerings        : {total_2025:,}")
print(f"Offerings with >= 1 Yr Lag : {has_lag1:,} ({has_lag1 / total_2025:.1%})")
print(f"Offerings with >= 2 Yrs Lag: {has_lag2:,} ({has_lag2 / total_2025:.1%})")
print(f"Offerings with >= 3 Yrs Lag: {has_lag3:,} ({has_lag3 / total_2025:.1%})")
print(f"Sparse History (< 3 Yrs)   : {total_2025 - has_lag3:,} ({(total_2025 - has_lag3) / total_2025:.1%})")


## 3. Cutoff Drift Dynamics (Year-over-Year Volatility)
Analyze how closing ranks shift year-over-year in log-space ($\Delta = \ln(C_t) - \ln(C_{t-1})$).


In [ ]:
drift_eval = df[df["prev_closing_1"].notna() & (df["prev_closing_1"] > 0) & (df["closing_rank"] > 0)].copy()
drift_eval["log_drift"] = np.log(drift_eval["closing_rank"]) - np.log(drift_eval["prev_closing_1"])

drift_by_tier = drift_eval.groupby("institute_type")["log_drift"].agg(["count", "mean", "std", "median"])
drift_by_tier["std_rank_percent"] = (np.exp(drift_by_tier["std"]) - 1.0) * 100
print("Annual Log-Cutoff Drift by Institute Tier:")
drift_by_tier


## 4. Category Scaling (CRL vs Category Ranks)
Examine closing rank scales across different quota seat pools (OPEN vs OBC-NCL vs SC vs ST).


In [ ]:
cat_summary = df_2025.groupby(["institute_type", "seat_type"])["closing_rank"].agg(["count", "median", "mean"]).reset_index()
cat_summary = cat_summary[cat_summary["count"] > 50].sort_values(["institute_type", "median"])
cat_summary.head(15)


## 5. Rank Distance to Admission Probability Mapping
Illustrate the normal cumulative distribution function mapping $P(	ext{admit}) = 1 - \Phi\left(rac{\ln(R) - \hat{y}}{\sigma}ight)$.


In [ ]:
z_vals = np.linspace(-3, 3, 100)
p_admit = 1.0 - norm.cdf(z_vals)

fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
ax.plot(z_vals, p_admit * 100, color="#2563EB", linewidth=2.5, label="P(Admit)")
ax.axvline(0, color="gray", linestyle="--", alpha=0.7, label="z = 0 (Expected Cutoff, P=50%)")
ax.axhline(80, color="#166534", linestyle=":", label="Safe Zone (P >= 80%)")
ax.axhline(30, color="#D97706", linestyle=":", label="Target Zone (30% <= P < 80%)")
ax.set_title("Mapping Standardized Rank Distance z to Admission Probability", fontweight="bold")
ax.set_xlabel("Normalized Distance z = (ln(Rank) - y_hat) / sigma")
ax.set_ylabel("Estimated Admission Odds (%)")
ax.grid(alpha=0.4)
ax.legend()
plt.tight_layout()
plt.close()
